# Search and Download NISAR GCOV Data with `asf_search`

<br>

[`asf_search`](https://github.com/asfadmin/Discovery-asf_search) is an open source Python package for searching and downloading SAR data from {abbr}`ASF (Alaska Satellite Facility)`. This notebook demonstrates how to search and download NISAR GCOV data with `asf_search`.

<hr>

## Overview

1. [Prerequisites](asf_search-prereqs)
1. [Search and download example](asf_search-download)
1. [Summary](asf_search-summary)
1. [Resources and references](asf_search-resources)

<hr>

(asf_search-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 3 minutes

<hr>

(asf_search-download)=
## 2. Search and download example

Search and download GCOV data with `asf_search` 

### 2a. Search for GCOV data with `asf_search.search()`

:::{tip} View all possible search parameters 

There are many `ASFSearchOptions` parameters, in addition to those used in the search below. A complete list can be found [here](https://github.com/asfadmin/Discovery-asf_search/blob/3db7664fce8c0e745623ac28e923594182cc1451/asf_search/ASFSearchOptions/validator_map.py#L41-L101).
:::

In [ ]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass

session = asf.ASFSession()
session.auth_with_creds(input('EDL Username'), getpass('EDL Password'))

start_date = datetime(2025, 11, 22)
end_date = datetime(2025, 12, 5)
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" # POINT or POLYGON as WKT (well-known-text)

opts=asf.ASFSearchOptions(**{
    "maxResults": 250,
    "intersectsWith": area_of_interest,
    "start": start_date,
    "end": end_date,
    "processingLevel": [
        "GCOV"
    ],
    "dataset": [
        "NISAR"
    ],
    "productionConfiguration": [
        "PR"
    ],
    'session': session,
})

response = asf.search(opts=opts)
hdf5_files = response.find_urls(extension='.h5', directAccess=False)
hdf5_files

### 2b. Filter the results

Note that the results include the `QA_STATS` files as well as the GCOV products. If you only want the GCOV data, you can remove the `QA_STATS` files with a pattern (regex) filter when calling `response.find_urls`

:::{tip} Reference the GCOV File Naming Convention
Refer to the File Naming Convention section (3.4) of the [NISAR L2 GCOV Product Specification](https://nisar.asf.earthdatacloud.nasa.gov/NISAR-SAMPLE-DATA/DOCS/NISAR_D-102274_RevE_NASA_SDS_Product_Specification_L2_GCOV_Nov8_2024_w-sigs.pdf) when creating your regex pattern.
:::

:::{tip} New to regex?
Here are some resources to get you started:
- [Regex Tutorial](https://www.geeksforgeeks.org/dsa/write-regular-expressions/)
- [Regex Sandbox](https://regex101.com/)
  
:::

In [ ]:
# Don't run this code cell if you wish to download the QA_STATS data

# Include include a negative lookahead for "QA_STATS"
pattern = r'^(?!.*QA_STATS).*'
# To include the static files, remove the `pattern=pattern` argument below
hdf5_files = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
hdf5_files

### 2c. Download the data to a specified directory

In [ ]:
from pathlib import Path

data_dir = Path.home() / "GCOV_data"
data_dir.mkdir(exist_ok=True)
print(f'data_dir: {data_dir}')

asf.download_urls(hdf5_files, data_dir, session=session, processes=4)

<hr>

(asf_search-summary)=
## 3. Summary
You now have the tools and knowledge that you need to search and download data using the `asf_search` Python package.

<hr>

(asf_search-resources)=
## 4. Resources and references
 - [asf_search](https://github.com/asfadmin/Discovery-asf_search)
 - [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/asf-search/)
 
**Author:** Alex Lewandowski